In [ ]:
# Install the ultralytics package
!pip install ultralytics -q

# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.2 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
!pip install ultralytics

import torch
if torch.cuda.is_available():
    print(f"GPU is good to go: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not found! Please check Step 1.")

GPU is good to go: Tesla T4


In [ ]:
import yaml
import os

# ⚠️ UPDATE THIS: Put the exact path to your dataset folder in your Drive
dataset_path = '/content/drive/MyDrive/yolov11/dataset'
yaml_file = os.path.join(dataset_path, 'data.yaml')

# Load existing yaml
with open(yaml_file, 'r') as f:
    data = yaml.safe_load(f)

# Update paths to point directly to your Drive folders
data['train'] = os.path.join(dataset_path, 'train/images')
data['val'] = os.path.join(dataset_path, 'valid/images')
# Note: Roboflow sometimes names the validation folder 'valid' or 'val'. Check your folder!

if 'test' in data:
    data['test'] = os.path.join(dataset_path, 'test/images')

# Save it back
with open(yaml_file, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print("data.yaml updated successfully!")

data.yaml updated successfully!


In [ ]:
import os

IMAGES_DIR = '/content/drive/MyDrive/yolov11/dataset/train/images'
LABELS_DIR = '/content/drive/MyDrive/yolov11/dataset/train/labels'
VEGETATION_CLASS_ID = 3  # UPDATE THIS to your vegetation class ID
SHRINK_FACTOR = 0.90

# 1. Tighten Vegetation Boxes
files_modified = 0
for label_file in os.listdir(LABELS_DIR):
    if not label_file.endswith('.txt'): continue
    label_path = os.path.join(LABELS_DIR, label_file)
    with open(label_path, 'r') as f: lines = f.readlines()

    new_lines = []
    modified = False
    for line in lines:
        parts = line.strip().split()
        if not parts: continue
        class_id = int(parts[0])

        if class_id != VEGETATION_CLASS_ID or len(parts) != 9:
            new_lines.append(line)
            continue

        coords = [float(x) for x in parts[1:]]
        points = [(coords[i], coords[i+1]) for i in range(0, 8, 2)]
        center_x = sum(p[0] for p in points) / 4.0
        center_y = sum(p[1] for p in points) / 4.0

        new_coords = []
        for (x, y) in points:
            new_x = max(0.0, min(1.0, center_x + (x - center_x) * SHRINK_FACTOR))
            new_y = max(0.0, min(1.0, center_y + (y - center_y) * SHRINK_FACTOR))
            new_coords.extend([new_x, new_y])

        new_lines.append(f"{class_id} " + " ".join([f"{val:.6f}" for val in new_coords]) + "\n")
        modified = True

    if modified:
        with open(label_path, 'w') as f: f.writelines(new_lines)
        files_modified += 1
print(f"✅ Tightened Vegetation OBBs in {files_modified} images.")

# 2. Create Background Labels
background_count = 0
for img_filename in os.listdir(IMAGES_DIR):
    if not img_filename.lower().endswith(('.jpg', '.jpeg', '.png')): continue
    label_path = os.path.join(LABELS_DIR, os.path.splitext(img_filename)[0] + '.txt')
    if not os.path.exists(label_path):
        with open(label_path, 'w') as f: pass
        background_count += 1
print(f"✅ Added {background_count} empty labels for healthy background images.")

✅ Tightened Vegetation OBBs in 62 images.
✅ Added 0 empty labels for healthy background images.


In [ ]:
import cv2
import albumentations as A

TARGET_CLASS_ID = 2  # UPDATE THIS to your Reverse Polarity class ID
COPIES_TO_MAKE = 10

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=1.0),
    A.RandomBrightnessContrast(p=0.5),
], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

files_processed, new_files_created = 0, 0

for label_file in os.listdir(LABELS_DIR):
    if not label_file.endswith('.txt'): continue
    label_path = os.path.join(LABELS_DIR, label_file)
    with open(label_path, 'r') as f: lines = f.readlines()

    if not any(int(line.split()[0]) == TARGET_CLASS_ID for line in lines if line.strip()):
        continue

    img_name = label_file.replace('.txt', '.jpg') # Change to .png if necessary
    img_path = os.path.join(IMAGES_DIR, img_name)
    if not os.path.exists(img_path): continue

    img = cv2.imread(img_path)
    img_h, img_w = img.shape[:2]
    all_classes, keypoints = [], []

    for line in lines:
        parts = line.strip().split()
        if len(parts) == 9:
            all_classes.append(int(parts[0]))
            coords = [float(x) for x in parts[1:]]
            for i in range(0, 8, 2):
                keypoints.append((coords[i] * img_w, coords[i+1] * img_h))

    files_processed += 1

    for copy_idx in range(COPIES_TO_MAKE):
        transformed = transform(image=img, keypoints=keypoints)
        cv2.imwrite(os.path.join(IMAGES_DIR, f"aug_{copy_idx}_{img_name}"), transformed['image'])

        with open(os.path.join(LABELS_DIR, f"aug_{copy_idx}_{label_file}"), 'w') as f:
            for i, class_id in enumerate(all_classes):
                box_points = transformed['keypoints'][i*4 : i*4+4]
                norm_coords = []
                for (x, y) in box_points:
                    norm_coords.extend([max(0.0, min(1.0, x/img_w)), max(0.0, min(1.0, y/img_h))])
                f.write(f"{class_id} " + " ".join([f"{v:.6f}" for v in norm_coords]) + "\n")
        new_files_created += 1

print(f"✅ Processed {files_processed} minority class images and generated {new_files_created} new augmented examples.")

✅ Processed 13 minority class images and generated 130 new augmented examples.


In [ ]:
import shutil

ORIGINAL_DATASET_DIR = '/content/drive/MyDrive/yolov11/dataset'
NEW_DATASET_DIR = '/content/drive/MyDrive/yolov11/dataset_clahe'
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def process_dataset_split(split_name):
    orig_img_dir = os.path.join(ORIGINAL_DATASET_DIR, split_name, 'images')
    orig_lbl_dir = os.path.join(ORIGINAL_DATASET_DIR, split_name, 'labels')
    new_img_dir = os.path.join(NEW_DATASET_DIR, split_name, 'images')
    new_lbl_dir = os.path.join(NEW_DATASET_DIR, split_name, 'labels')

    if not os.path.exists(orig_img_dir): return
    os.makedirs(new_img_dir, exist_ok=True)
    os.makedirs(new_lbl_dir, exist_ok=True)

    for lbl_file in os.listdir(orig_lbl_dir):
        if lbl_file.endswith('.txt'):
            shutil.copy(os.path.join(orig_lbl_dir, lbl_file), os.path.join(new_lbl_dir, lbl_file))

    img_count = 0
    for img_file in os.listdir(orig_img_dir):
        if not img_file.lower().endswith(('.png', '.jpg', '.jpeg')): continue
        img = cv2.imread(os.path.join(orig_img_dir, img_file))
        if img is None: continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe_img = clahe.apply(gray)
        final_img = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR) # YOLO needs 3 channels

        cv2.imwrite(os.path.join(new_img_dir, img_file), final_img)
        img_count += 1
    print(f"✅ Finished '{split_name}': Processed {img_count} images into high-contrast.")

print("🚀 Starting CLAHE Pipeline...")
for split in ['train', 'valid', 'test']: process_dataset_split(split)
print(f"🎉 Dataset safely duplicated and enhanced in: {NEW_DATASET_DIR}")

🚀 Starting CLAHE Pipeline...
✅ Finished 'train': Processed 291 images into high-contrast.
✅ Finished 'valid': Processed 48 images into high-contrast.
✅ Finished 'test': Processed 21 images into high-contrast.
🎉 Dataset safely duplicated and enhanced in: /content/drive/MyDrive/yolov11/dataset_clahe


In [ ]:
import shutil
import os

# Paths
old_yaml_path = '/content/drive/MyDrive/yolov11/dataset/data.yaml'
new_yaml_path = '/content/drive/MyDrive/yolov11/dataset_clahe/data.yaml'

# Copy the file
if os.path.exists(old_yaml_path):
    shutil.copy(old_yaml_path, new_yaml_path)
    print("✅ data.yaml successfully copied to the new folder!")
else:
    print("❌ Could not find data.yaml in the original folder. Please check the path.")

✅ data.yaml successfully copied to the new folder!


In [ ]:
from ultralytics import YOLO

# Load the Medium OBB Model
model = YOLO('yolo11m-obb.pt')

# Train using the NEW CLAHE dataset and benchmark hyperparameters
results = model.train(
    data='/content/drive/MyDrive/yolov11/dataset_clahe/data.yaml', # Note the new path!

    epochs=50,
    patience=50,
    imgsz=1024,             # High resolution for tiny anomalies
    batch=16,               # Lower to 8 if Colab runs out of GPU memory

    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,

    mosaic=1.0,
    mixup=0.15,
    degrees=15.0,
    close_mosaic=15,

    project='/content/drive/MyDrive/YOLO_Results',
    name='solar_anomaly_final_run',
    save_period=10
)

print("🎉 Training complete! Your highly optimized model is ready.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/yolov11/dataset_clahe/data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7,

In [ ]:
from ultralytics import YOLO

# ==========================================
# 1. LOAD THE BEST MODEL
# ==========================================
# Point this to the 'best.pt' file inside your most successful run's weights folder
best_model_path = '/content/drive/MyDrive/YOLO_Results/solar_anomaly_final_run/weights/best.pt'
model = YOLO(best_model_path)

# ==========================================
# 2. RUN EVALUATION ON UNSEEN TEST DATA
# ==========================================
print("🚀 Starting Evaluation on the Unseen Test Set...")
metrics = model.val(
    data='/content/drive/MyDrive/yolov11/dataset_clahe/data.yaml',
    split='test',  # 'test' ensures the model has never seen these images
    task='obb',    # Oriented Bounding Box task
    plots=True     # Automatically generates Confusion Matrix, F1-Curve, etc.
)

# ==========================================
# 3. OUTPUT METRICS PROGRAMMATICALLY
# ==========================================
print("\n" + "="*50)
print("🏆 FINAL TEST SET METRICS (OVERALL) 🏆")
print("="*50)

# General Object Detection Metrics
print(f"Overall Precision:             {metrics.box.mp:.4f}")
print(f"Overall Recall:                {metrics.box.mr:.4f}")
print(f"mAP @ 50% IoU (mAP50):         {metrics.box.map50:.4f}")
print(f"mAP @ 50-95% IoU (mAP50-95):   {metrics.box.map:.4f}")

print("\n" + "="*50)
print("📊 PER-CLASS METRICS (mAP50-95) 📊")
print("="*50)

# Loop through and print individual class metrics
class_names = metrics.names
for i, class_id in enumerate(metrics.box.ap_class_index):
    class_name = class_names[class_id]

    # FIXED: Using .ap and .ap50 instead of the old .maps and .map50s
    class_map = metrics.box.ap[i]      # This is the per-class mAP50-95
    class_map50 = metrics.box.ap50[i]  # This is the per-class mAP50

    print(f"🔹 {class_name:<20} | mAP50: {class_map50:.4f} | mAP50-95: {class_map:.4f}")

print("\n" + "="*50)
print(f"✅ Evaluation complete! Check the folder: {metrics.save_dir}")
print("Look for files like 'confusion_matrix.png' and 'PR_curve.png' to see exactly where the model gets confused.")

🚀 Starting Evaluation on the Unseen Test Set...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m-obb summary (fused): 135 layers, 20,881,567 parameters, 0 gradients, 71.4 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.3 ms, read: 29.8±10.4 MB/s, size: 48.1 KB)
val: Scanning /content/drive/MyDrive/yolov11/dataset/test/labels.cache... 21 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 21/21 6.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.7s/it 5.4s
                   all         21         64      0.902      0.876      0.904      0.762
         Diode anomaly         10         16      0.825      0.884      0.906      0.852
             Hot Spots         13         25      0.782       0.86      0.829      0.754
      Reverse polarity          1          2          1          1      0.995      0.895
            Vegetation          7         21          1      0.761   